# GPX1 Discovery Agent
## 02 — Active Learning

Goal: simulate a closed-loop discovery campaign with a fixed budget of 40 prospective experiments.

Each campaign begins with a small historical set, scores the untested pool, selects one compound, reveals its hidden activity label, retrains the surrogate, and repeats.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

PROJECT_ROOT = Path("..")
sys.path.append(str(PROJECT_ROOT))

from src.featurization import parse_smiles, morgan_fingerprints
from src.campaign import scaffold_split, create_seed_set
from src.environment import GPX1DiscoveryEnv
from src.evaluation import completed_campaign_metrics
from src.models import fit_surrogate

RANDOM_SEED = 42
BUDGET = 40

df = pd.read_csv(PROJECT_ROOT / "data" / "GPX1_curated_for_RL.csv")
mols, valid_mask = parse_smiles(df["PUBCHEM_EXT_DATASOURCE_SMILES"])
clean = df.loc[valid_mask].copy().reset_index(drop=True)

X = morgan_fingerprints(mols)
y = clean["label"].to_numpy(dtype=int)
scaffolds = clean["scaffold"].astype(str).to_numpy()

campaign_idx, validation_idx = scaffold_split(
    X, y, scaffolds,
    test_size=0.20,
    random_state=RANDOM_SEED,
)

X_campaign = X[campaign_idx]
y_campaign = y[campaign_idx]
campaign_scaffolds = scaffolds[campaign_idx]

X_validation = X[validation_idx]
y_validation = y[validation_idx]

### Historical seed

Each campaign starts with **5 known actives + 45 known inactives**. This represents prior discovery knowledge and is not counted against the prospective budget.

The seed is intentionally stratified because a random 50-compound seed at ~1.7% prevalence frequently contains no positives.

In [ ]:
seed_idx, pool_idx = create_seed_set(
    y_campaign,
    campaign_seed=42,
    n_actives=5,
    n_inactives=45,
)

print("Historical labels:", len(seed_idx))
print("Historical actives:", int(y_campaign[seed_idx].sum()))
print("Prospective pool:", len(pool_idx))

### Acquisition policies

- **Exploit**: select the highest predicted activity score.
- **Explore**: select the highest predictive entropy.
- **ε-greedy**: explore with probability ε, otherwise exploit.

The uncertainty signal here is **decision uncertainty**, not rigorous epistemic uncertainty.

In [ ]:
def run_policy(campaign_seed, mode, epsilon=0.0):
    seed_idx, pool_idx = create_seed_set(
        y_campaign,
        campaign_seed,
    )
    rng = np.random.default_rng(campaign_seed)

    env = GPX1DiscoveryEnv(
        X_campaign,
        y_campaign,
        campaign_scaffolds,
        budget=BUDGET,
        activity_reward=0.0,
        novelty_reward=0.0,
        information_reward=0.0,
    )
    env.reset(seed_idx, pool_idx)

    while True:
        if mode == "exploit":
            action = 0
        elif mode == "explore":
            action = 1
        elif mode == "epsilon":
            action = 1 if rng.random() < epsilon else 0
        else:
            raise ValueError(mode)

        _, _, done, _ = env.step(action)
        if done:
            break

    history = pd.DataFrame(env.history)

    metrics = completed_campaign_metrics(
        history,
        seed_idx,
        X_campaign,
        y_campaign,
        campaign_scaffolds,
        X_validation,
        y_validation,
        fit_surrogate,
    )
    metrics["explore_actions"] = int(
        (history["action"] == 1).sum()
    )
    return metrics

### Repeated ε-greedy evaluation

In [ ]:
rows = []

for epsilon in [0.0, 0.1, 0.2, 0.3, 0.5, 1.0]:
    for campaign_seed in range(10, 20):
        metrics = run_policy(
            campaign_seed,
            mode="epsilon",
            epsilon=epsilon,
        )
        rows.append({
            "epsilon": epsilon,
            "campaign": campaign_seed,
            **metrics,
        })

results = pd.DataFrame(rows)

summary = (
    results
    .groupby("epsilon")
    .agg(
        mean_hits=("hits", "mean"),
        hit_sd=("hits", "std"),
        min_hits=("hits", "min"),
        median_hits=("hits", "median"),
        mean_active_scaffolds=("active_scaffolds", "mean"),
        mean_validation_pr_auc=("validation_pr_auc", "mean"),
    )
    .reset_index()
)

summary

### Design note: discarded additive score

An early acquisition rule added activity score and uncertainty directly. A suspicious abrupt optimum at λ=0.5 was investigated analytically and found to be a scoring degeneracy: for predictions above 0.5, the acquisition score collapses to a constant at λ=0.5.

That formulation was removed rather than reported as a successful hyperparameter result.

## Takeaway

In this dataset, pure exploitation is a very strong discovery baseline. Increasing uncertainty-driven exploration sacrifices immediate hits and yields only modest gains in final model performance. This motivates an explicit scientific reward rather than assuming exploration is intrinsically valuable.